In [65]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('.')))

from sqlalchemy import text
from db.database import engine
from etl.constants import EUROPEAN_COUNTRIES, DECOUPLING_START_YEAR, DECOUPLING_END_YEAR

In [66]:
query = """
    SELECT 
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_total,
        e.co2_per_capita,
        e.co2_per_gdp,
        e.consumption_co2,
        e.consumption_co2_per_capita,
        e.consumption_co2_per_gdp,
        e.trade_co2,
        e.trade_co2_share,
        e.gdp,
        e.population,
        e.coal_co2,
        e.gas_co2,
        e.oil_co2
    FROM 
        emissions e
    JOIN 
        countries c ON c.id = e.country_id
    WHERE 
        e.year BETWEEN :start AND :end
    ORDER BY 
        c.iso_code, 
        e.year
"""

with engine.connect() as conn:
    df = pd.read_sql(
        text(query),
        conn,
        params={
            'start': DECOUPLING_START_YEAR,
            'end': DECOUPLING_END_YEAR
        }
    )

df_eu = df[df['iso_code'].isin(EUROPEAN_COUNTRIES)].copy()

print(f"Global dataset: {len(df)} rows, {df['iso_code'].nunique()} countries")
print(f"European dataset: {len(df_eu)} rows, {df_eu['iso_code'].nunique()} countries")

2026-06-12 00:27:00,037 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-12 00:27:00,039 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2026-06-12 00:27:00,043 INFO sqlalchemy.engine.Engine [cached since 3286s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x000001D3ECD11F50>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-06-12 00:27:00,047 INFO sqlalchemy.engine.Engine 
    SELECT 
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_total,
        e.co2_per_capita

In [67]:
def build_index(df: pd.DataFrame, base_year: int = DECOUPLING_START_YEAR) -> pd.DataFrame:
    # Normalize GDP and CO2 to base_year = 100.
    df = df.copy()
    
    results = []
    for _, group in df.groupby('iso_code'):
        base = group[group['year'] == base_year]
        if base.empty:
            continue
        
        base_gdp = base['gdp'].values[0]
        base_co2 = base['co2_total'].values[0]
        base_consumption = base['consumption_co2'].values[0]
        
        if pd.isna(base_gdp) or pd.isna(base_co2):
            continue
            
        group = group.copy()
        group['gdp_index'] = group['gdp'] / base_gdp * 100
        group['co2_index'] = group['co2_total'] / base_co2 * 100
        
        if not pd.isna(base_consumption):
            group['consumption_co2_index'] = group['consumption_co2'] / base_consumption * 100
        else:
            group['consumption_co2_index'] = np.nan
            
        results.append(group)
    
    return pd.concat(results, ignore_index=True)

df_indexed = build_index(df)
df_eu_indexed = build_index(df_eu)

print(f"Indexed dataset: {len(df_eu_indexed)} rows")
print(f"Sample - Poland 1995, 2000, 2005, 2010, 2015, 2020:")
print(df_eu_indexed[df_eu_indexed['iso_code'] == 'POL'][
    ['year', 'gdp_index', 'co2_index', 'consumption_co2_index']
].query('year in [1995, 2000, 2005, 2010, 2015, 2020]').to_string())

Indexed dataset: 1156 rows
Sample - Poland 1995, 2000, 2005, 2010, 2015, 2020:
     year   gdp_index  co2_index  consumption_co2_index
923  1995  116.178057  96.317445              99.898607
928  2000  157.145110  84.313204              90.634862
933  2005  191.828134  85.713792              91.070020
938  2010  256.095460  88.756503             101.600715
943  2015  301.295355  83.049195              92.169933
948  2020  353.613200  80.273262              88.802083


In [68]:
# Finding countries with growing GDP and falling territorial CO2

latest = df_eu_indexed[df_eu_indexed['year'] == 2022].copy()
latest = latest.dropna(subset=['gdp_index', 'co2_index'])

# Classify countries
def classify_decoupling(row):
    gdp_grew = row['gdp_index'] > 120
    co2_fell = row['co2_index'] < 95
    co2_stable = row['co2_index'] < 110
    
    if gdp_grew and co2_fell:
        return 'Strong decoupling'
    elif gdp_grew and co2_stable:
        return 'Weak decoupling'
    elif gdp_grew:
        return 'No decoupling'
    else:
        return 'Economy shrank'

latest['decoupling_type'] = latest.apply(classify_decoupling, axis=1)

fig = px.scatter(
    latest,
    x='gdp_index',
    y='co2_index',
    color='decoupling_type',
    text='iso_code',
    title='Territorial Decoupling - European Countries (1990 = 100, measured at 2022)',
    labels={
        'gdp_index': 'GDP Index (1990 = 100)',
        'co2_index': 'CO2 Index (1990 = 100)',
        'decoupling_type': 'Decoupling type',
    },
    color_discrete_map={
        'Strong decoupling': 'green',
        'Weak decoupling': 'orange',
        'No decoupling': 'red',
        'Economy shrank': 'gray'
    },
    height=600
)
fig.add_hline(y=100, line_dash='dash', line_color='gray', annotation_text='CO2 baseline (1990)')
fig.add_vline(x=100, line_dash='dash', line_color='gray', annotation_text='GDP baseline (1990)')
fig.add_annotation(
    x=260, 
    y=33,
    text="Green quadrant<br>GDP up, CO2 down",
    showarrow=False,
    font=dict(color='green')
)
fig.update_traces(textposition='top center')
fig.show()

print("\nDecoupling classification:")
print(latest.groupby('decoupling_type')['country'].apply(list))


Decoupling classification:
decoupling_type
Economy shrank                                      [Georgia, Ukraine]
No decoupling                                [Cyprus, Ireland, Norway]
Strong decoupling    [Albania, Belgium, Bulgaria, Belarus, Switzerl...
Weak decoupling                                       [Austria, Spain]
Name: country, dtype: object


In [69]:
# Pick top 8 decouplers + Poland for context
top_decouplers = latest[latest['decoupling_type'] == 'Strong decoupling']['iso_code'].tolist()

# Always include Poland for context
focus_countries = top_decouplers[:8] + ['POL']
focus_countries = list(set(focus_countries))

df_focus = df_eu_indexed[df_eu_indexed['iso_code'].isin(focus_countries)]

fig = px.line(
    df_focus,
    x='year',
    y='co2_index',
    color='country',
    title='CO2 Trajectory (1990 = 100) - Top Decouplers',
    labels={
        'co2_index': 'CO2 Index (1990 = 100)', 
        'year': 'Year',
        'country': 'Country',
        },
    height=500
)
fig.add_hline(y=100, line_dash='dash', line_color='gray', annotation_text='Territorial CO2 baseline (1990)')
fig.show()

fig_gdp = px.line(
    df_focus,
    x='year',
    y='gdp_index',
    color='country',
    title='GDP Trajectory (1990 = 100) - Top Decouplers',
    labels={
        'gdp_index': 'GDP Index (1990 = 100)', 
        'year': 'Year',
        'country': 'Country'
        },
    height=500
)
fig_gdp.add_hline(y=100, line_dash='dash', line_color='gray',  annotation_text='GDP baseline (1990)')
fig_gdp.show()

In [70]:
latest_full = df_eu_indexed[
    (df_eu_indexed['year'] == 2022) &
    (df_eu_indexed['consumption_co2_index'].notna())
].copy()

latest_full['gap'] = latest_full['co2_index'] - latest_full['consumption_co2_index']
# Positive gap = territorial looks better than reality (imported emissions)
# Negative gap = territorial looks worse than reality (exported emissions)

latest_full = latest_full.sort_values('gap', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Territorial<br>CO2 index (emit)',
    x=latest_full['country'],
    y=latest_full['co2_index'],
    marker_color='steelblue'
))
fig.add_trace(go.Bar(
    name='Consumption<br>CO2 index<br>(consume)',
    x=latest_full['country'],
    y=latest_full['consumption_co2_index'],
    marker_color='coral'
))
fig.add_hline(y=100, line_dash='dash', line_color='gray', annotation_text='CO2 baseline (1990)')
fig.update_layout(
    title='Territorial CO2 vs Consumption CO2 in 2022<br><sup>Difference between what countries emit and what they consume</sup>',
    barmode='group',
    height=600,
    xaxis_tickangle=-45
)
fig.show()

print("\nThe healthiest ones (consumption co2 smaller than territorial co2):")
print(latest_full[latest_full['gap'] > 2][['country', 'co2_index', 'consumption_co2_index', 'gap']].to_string(index=False))

print("\nCountries that consume more than emit ('fake decouplers'):")
print(latest_full[latest_full['gap'] < -30][['country', 'co2_index', 'consumption_co2_index', 'gap']].to_string(index=False))


The healthiest ones (consumption co2 smaller than territorial co2):
country  co2_index  consumption_co2_index      gap
 Sweden  63.294843              59.142009 4.152834
Finland  63.837599              59.534690 4.302910
Austria  98.814941              93.802594 5.012347

Countries that consume more than emit ('fake decouplers'):
    country  co2_index  consumption_co2_index         gap
      Malta  73.094355             493.993258 -420.898903
    Belgium  73.889558             172.101217  -98.211659
    Georgia  80.808081             164.234857  -83.426776
Switzerland  74.635892             144.851712  -70.215820
    Belarus  52.665984             104.387680  -51.721696
    Croatia  76.840725             118.812226  -41.971501
     Latvia  33.607975              68.117394  -34.509420
    Denmark  54.404697              86.629132  -32.224435


In [71]:
def plot_three_trajectories(iso_codes: list, df_indexed: pd.DataFrame, title: str):
    # For each country show GDP, territorial CO2 and consumption CO2
    # on the same chart (1990 = 100)
    fig = make_subplots(
        rows=len(iso_codes) // 2 + len(iso_codes) % 2,
        cols=2,
        subplot_titles=[
            df_indexed[df_indexed['iso_code'] == iso]['country'].iloc[0]
            for iso in iso_codes
        ]
    )

    for i, iso in enumerate(iso_codes):
        row = i // 2 + 1
        col = i % 2 + 1

        country_df = df_indexed[
            df_indexed['iso_code'] == iso
        ].dropna(subset=['gdp_index'])

        fig.add_trace(go.Scatter(
            x=country_df['year'],
            y=country_df['gdp_index'],
            name='GDP',
            line=dict(color='blue', width=2),
            showlegend=(i == 0)
        ), row=row, col=col)

        fig.add_trace(go.Scatter(
            x=country_df['year'],
            y=country_df['co2_index'],
            name='Territorial CO2',
            line=dict(color='red', width=2),
            showlegend=(i == 0)
        ), row=row, col=col)

        fig.add_trace(go.Scatter(
            x=country_df['year'],
            y=country_df['consumption_co2_index'],
            name='Consumption CO2',
            line=dict(color='orange', width=2, dash='dash'),
            showlegend=(i == 0)
        ), row=row, col=col)

        fig.add_hline(
            y=100,
            line_dash='dot',
            line_color='gray',
            row=row, col=col
        )

    fig.update_layout(
        height=310 * (len(iso_codes) // 2 + 1),
        title_text=title
    )
    fig.show()

importers = ['GBR', 'DEU', 'FRA', 'ITA', 'CHE', 'BEL', 'DNK', 'NLD']
plot_three_trajectories(
    importers,
    df_eu_indexed,
    'Fake decoupling test - biggest emission importers<br>'
    '<sup>If consumption CO2 (orange) stays high while territorial (red) drops - fake decoupling</sup><br>'
)

In [72]:
# Best territorial performers  is their decoupling real?
best_performers = ['LTU', 'SWE', 'ROU', 'SVK']
plot_three_trajectories(
    best_performers,
    df_eu_indexed,
    'Real decoupling test - best territorial performers<br>'
    '<sup>If consumption CO2 (orange) also drops - decoupling is real</sup>'
)

In [73]:
# Interesting outliers
outliers = ['POL', 'IRL', 'CYP', 'GEO']
plot_three_trajectories(
    outliers,
    df_eu_indexed,
    'Outliers and interesting cases'
)

In [74]:
# Special one for Norway
query = """
    SELECT 
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_total,
        e.consumption_co2,
        e.gdp,
        e.population
    FROM 
        emissions e
    JOIN 
        countries c ON c.id = e.country_id
    WHERE 
        e.year BETWEEN :start AND :end AND c.iso_code = 'NOR'
    ORDER BY 
        c.iso_code, 
        e.year
"""

with engine.connect() as conn:
    df_nor = pd.read_sql(
        text(query),
        conn,
        params={
            'start': 2003,
            'end': DECOUPLING_END_YEAR
        }
    )

nor = build_index(df_nor, 2003).copy()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=nor['year'], 
    y=nor['gdp_index'],
    name='GDP', 
    line=dict(color='blue', width=2)
))
fig.add_trace(go.Scatter(
    x=nor['year'], 
    y=nor['co2_index'],
    name='Territorial CO2', 
    line=dict(color='red', width=2)
))
fig.add_trace(go.Scatter(
    x=nor['year'], 
    y=nor['consumption_co2_index'],
    name='Consumption CO2', 
    line=dict(color='orange', width=2, dash='dash')
))
fig.add_hline(y=100, line_dash='dot', line_color='gray')

fig.update_layout(
    height=500,
    title_text='Decopling test for Norway (2003 = 100)<br>'
               '<sup>*cause for consumption co2 Norway has data only from 2003</sup>'
)
fig.show()

2026-06-12 00:27:03,614 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-12 00:27:03,615 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2026-06-12 00:27:03,617 INFO sqlalchemy.engine.Engine [cached since 3289s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x000001D3E6B69610>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-06-12 00:27:03,621 INFO sqlalchemy.engine.Engine 
    SELECT 
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_total,
        e.consumption_co

In [75]:
# trade_co2 positive = net importer of emissions
# trade_co2 negative = net exporter of emissions

df_trade = df_eu[
    (df_eu['year'] >= 1990) &
    (df_eu['trade_co2'].notna())
].copy()

latest_trade = df_trade[df_trade['year'] == df_trade['year'].max()].copy()
latest_trade = latest_trade.sort_values('trade_co2', ascending=True)

colors = ['green' if x < 0 else 'red' for x in latest_trade['trade_co2']]

fig = go.Figure(go.Bar(
    x=latest_trade['country'],
    y=latest_trade['trade_co2'],
    marker_color=colors,
    # text=latest_trade['country'],
))

fig.add_hline(y=0, line_color='black')
fig.update_layout(
    title='Net Trade CO2 - European Countries<br><sup>Negative = exporting emissions | Positive = importing emissions</sup>',
    yaxis_title='Net CO2 from trade (million tonnes)',
    height=500,
    xaxis_tickangle=-45
)
fig.show()

# Trade CO2 as % of total
latest_trade['trade_impact'] = (latest_trade['trade_co2'] / latest_trade['co2_total'] * 100)

print("\nTrade CO2 as % of territorial emissions:")
print(latest_trade[['country', 'trade_co2', 'trade_co2_share', 'trade_impact']]
      .sort_values('trade_impact').round(1).to_string(index=False))


Trade CO2 as % of territorial emissions:
       country  trade_co2  trade_co2_share  trade_impact
        Poland       -9.4             -3.3          -3.3
        Norway        0.1              0.3           0.3
      Bulgaria        0.8              2.3           2.3
    Luxembourg        0.5              7.2           7.2
       Belarus        6.8             12.2          12.2
      Slovakia        4.3             13.9          13.9
       Romania       10.4             15.3          15.3
       Ukraine       27.0             19.4          19.4
        Cyprus        1.4             20.0          20.0
       Czechia       16.8             20.2          20.2
         Spain       45.2             21.0          21.0
       Ireland        7.3             21.6          21.6
       Albania        1.0             21.8          21.8
      Portugal        9.8             26.1          26.1
   Netherlands       33.1             28.3          28.3
       Croatia        5.2             28.4    

In [76]:
# Trade CO2 over time
top_importers = ['GBR', 'DEU', 'FRA', 'ITA', 'CHE']
top_exporters = ['POL']  # only real exporter of co2 in Europe
focus = top_importers + top_exporters

df_trade_time = df_eu[
    (df_eu['iso_code'].isin(focus)) &
    (df_eu['trade_co2'].notna())
].copy()

fig = px.line(
    df_trade_time,
    x='year',
    y='trade_co2',
    color='country',
    title='Trade CO2 Over Time - Importers vs Poland<br>'
          '<sup>Positive = importing emissions | Negative = exporting</sup>',
    labels={
        'trade_co2': 'Net CO2 from trade (Mt)', 
        'year': 'Year',
        'country': 'Country'
    },
    height=500
)
fig.add_hline(y=0, line_dash='dash', line_color='black', annotation_text='Zero emissions line')

fig.add_annotation(
    x=2007, y=-7,
    text="Poland - only one net<br>exporter in Europe",
    showarrow=True,
    arrowhead=2,
    arrowwidth=2,
    font=dict(color='green', weight='bold')
)
fig.show()

In [77]:
deu = df_indexed[df_indexed['iso_code'] == 'DEU'].copy()

fig = make_subplots(
    rows=2, 
    cols=2,
    subplot_titles=[
        'Three trajectories (1990 = 100)',
        'Absolute: Territorial vs Consumption CO2',
        'Energy mix - what drove the reduction?',
        'Trade CO2 - growing import dependency'
    ]
)

fig.add_trace(go.Scatter(x=deu['year'], y=deu['gdp_index'],
    name='GDP', line=dict(color='blue', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=deu['year'], y=deu['co2_index'],
    name='Territorial CO2', line=dict(color='red', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=deu['year'], y=deu['consumption_co2_index'],
    name='Consumption CO2', line=dict(color='orange', width=2, dash='dash')), row=1, col=1)
fig.add_hline(y=100, line_dash='dot', line_color='gray', row=1, col=1)

fig.add_trace(go.Scatter(x=deu['year'], y=deu['co2_total'],
    name='Territorial (Mt)', line=dict(color='red'),
    showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=deu['year'], y=deu['consumption_co2'],
    name='Consumption (Mt)', line=dict(color='orange', dash='dash'),
    showlegend=False), row=1, col=2)

fig.add_trace(go.Scatter(x=deu['year'], y=deu['coal_co2'],
    name='Coal', line=dict(color='black')), row=2, col=1)
fig.add_trace(go.Scatter(x=deu['year'], y=deu['gas_co2'],
    name='Gas', line=dict(color='orange')), row=2, col=1)
fig.add_trace(go.Scatter(x=deu['year'], y=deu['oil_co2'],
    name='Oil', line=dict(color='brown')), row=2, col=1)

fig.add_trace(go.Scatter(x=deu['year'], y=deu['trade_co2'],
    name='Trade CO2', line=dict(color='purple'),
    showlegend=False), row=2, col=2)
fig.add_hline(y=0, line_dash='dot', line_color='gray', row=2, col=2)

fig.update_layout(
    height=700,
    title_text='Germany - Honest Decoupler?<br>'
               '<sup>Both territorial and consumption CO2 dropped</sup>'
)
fig.show()

deu_1990 = deu[deu['year'] == 1990].iloc[0]
deu_2022 = deu[deu['year'] == 2022].iloc[0]
print(f"Germany 1990-2022:")
print(f"  GDP grew:              {deu_2022['gdp_index']:.0f} (index, 1990 = 100)")
print(f"  Territorial CO2 fell:  {deu_2022['co2_index']:.0f} (index, 1990 = 100)")
print(f"  Consumption CO2 fell:  {deu_2022['consumption_co2_index']:.0f} (index, 1990 = 100)")
print(f"  Trade CO2 2022:        {deu_2022['trade_co2']:.0f} Mt (importing)")
print(f"\n Germany's decoupling is mostly real, but some emissions are being outsourced and still are")

Germany 1990-2022:
  GDP grew:              194 (index, 1990 = 100)
  Territorial CO2 fell:  63 (index, 1990 = 100)
  Consumption CO2 fell:  70 (index, 1990 = 100)
  Trade CO2 2022:        171 Mt (importing)

 Germany's decoupling is mostly real, but some emissions are being outsourced and still are


In [78]:
# Poland - the misunderstood emitter
pol = df_indexed[df_indexed['iso_code'] == 'POL'].copy()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Three trajectories (1990 = 100)',
        'Trade CO2 - Poland exports emissions',
        'Energy mix - still coal-heavy',
        'Consumption vs Territorial gap over time'
    ]
)

fig.add_trace(go.Scatter(x=pol['year'], y=pol['gdp_index'],
    name='GDP', line=dict(color='blue', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=pol['year'], y=pol['co2_index'],
    name='Territorial CO2', line=dict(color='red', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=pol['year'], y=pol['consumption_co2_index'],
    name='Consumption CO2', line=dict(color='orange', width=2, dash='dash')), row=1, col=1)
fig.add_hline(y=100, line_dash='dot', line_color='gray', row=1, col=1)

colors = ['green' if x < 0 else 'red' for x in pol['trade_co2'].fillna(0)]
fig.add_trace(go.Bar(
    x=pol['year'], y=pol['trade_co2'],
    name='Trade CO2',
    marker_color=colors,
    showlegend=False
), row=1, col=2)
fig.add_hline(y=0, line_dash='dot', line_color='black', row=1, col=2)

fig.add_trace(go.Scatter(x=pol['year'], y=pol['coal_co2'],
    name='Coal', line=dict(color='black')), row=2, col=1)
fig.add_trace(go.Scatter(x=pol['year'], y=pol['gas_co2'],
    name='Gas', line=dict(color='orange')), row=2, col=1)
fig.add_trace(go.Scatter(x=pol['year'], y=pol['oil_co2'],
    name='Oil', line=dict(color='brown')), row=2, col=1)

pol['gap'] = pol['co2_total'] - pol['consumption_co2']
fig.add_trace(go.Scatter(
    x=pol['year'], 
    y=pol['gap'],
    name='Gap (territorial - consumption)',
    line=dict(color='green', width=2),
    fill='tozeroy',
    fillcolor='rgba(0,255,0,0.1)',
    showlegend=False
), row=2, col=2)
fig.add_hline(y=0, line_dash='dot', line_color='black', row=2, col=2)

fig.update_layout(
    height=700,
    title_text='Poland - The Misunderstood Emitter?<br>'
               '<sup>GDP grew 4x, emissions barely moved - '
               'and Poland actually exports more emissions than it imports</sup>'
)
fig.show()

pol_2022 = pol[pol['year'] == 2022].iloc[0]
pol_1990 = pol[pol['year'] == 1990].iloc[0]
print(f"Poland 1990-2022:")
print(f"  GDP grew:              {pol_2022['gdp_index']:.0f} (index, 1990 = 100)")
print(f"  Territorial CO2:       {pol_2022['co2_index']:.0f} (index, barely moved)")
print(f"  Consumption CO2:       {pol_2022['consumption_co2_index']:.0f} (index, slightly lower than territorial)")
print(f"  Trade CO2 2022:        {pol_2022['trade_co2']:.0f} Mt")
print(f"\nPoland is Europe's manufacturing floor - its 'dirty' stats")
print(f"partly reflect goods consumed by richer neighbors")

Poland 1990-2022:
  GDP grew:              396 (index, 1990 = 100)
  Territorial CO2:       84 (index, barely moved)
  Consumption CO2:       92 (index, slightly lower than territorial)
  Trade CO2 2022:        -15 Mt

Poland is Europe's manufacturing floor - its 'dirty' stats
partly reflect goods consumed by richer neighbors


In [79]:
print("=" * 27)
print("NOTEBOOK 04 - KEY FINDINGS")
print("=" * 27)

print("""
Finding 1: Territorial decoupling - widespread but uneven
   27/34 European countries show strong territorial decoupling.
   
   Most impressive performers (CO2 index, GDP index):
    Estonia:   CO2 (-68%), GDP (+48%)
    Lithuania: CO2 (-64%), GDP (+66%)
    Romania:   CO2 (-59%), GDP (+294%) - standout
    Slovakia:  CO2 (-49%), GDP (+131%)
    Sweden:    CO2 (-42%), GDP (+105%)
   
   Notable: Romania grew its economy nearly 4x while cutting
   emissions by 60% - the strongest combined performance in Europe.

Finding 2: Real vs Fake decoupling - the honest test
   Key metric: trade_co2_share - how much of consumption
   is actually imported emissions.
   
   Fake decouplers (consumption stayed high):
    Malta        - trade impact 709% - imports almost all consumption
    Switzerland  - trade impact 270% - most extreme large-country fake decoupler
    Belgium      - trade impact 132% - heavy outsourcing
    Belarus      - obvious faker, without doubt
    Denmark      - trade impact 72% - significant gap
    France       - consumption CO2 lagged territorial by ~15 years
    UK           - partial fake decoupling until ~2010, then genuine

   
   Genuine decouplers (both curves dropped, small trade gap):
    Germany   - trade impact 29%, gap only -6.6 - most honest large economy
    Romania   - trade impact 15%, gap only -10  - genuine reduction
    Slovakia  - trade impact 14%, gap only -1.8  - nearly perfect alignment
    Bulgaria  - trade impact  2%, gap only -6.6 - genuine decoupler
    Sweden    - gap +4 - consumption CO2 lower than territorial,
                truly clean economy
    Finland - phenomenal decoupling (but only from 2010)
      
Finding 3: Germany - Europe's most honest large-economy decoupler
   Despite being Europe's largest absolute emitter:
    Territorial CO2 fell to 63 (index, 1990 = 100)
    Consumption CO2 fell to 70 - gap of only 6.6 points
    Trade impact 29% - growing but not undermining the story
    Coal CO2 dropped from 560Mt to ~200Mt - Energiewende is real
    Germany's green reputation is largely deserved.

Finding 4: Poland - Europe's misunderstood manufacturing floor
    GDP index: 396 - economy grew nearly 4x
    Territorial CO2: 84 - barely changed
    Consumption CO2: 92 - nearly identical to territorial
    Trade CO2: negative (-3.3% impact) - only one net exporter in Europe
    Poland manufactures goods consumed by richer neighbors. Its 'dirty' stats 
      partly reflect Western European consumption. Coal dependence is real, but
      Poland is Europe's most honest emissions reporter - what it produces, it consumes.

Finding 5: Norway - the late and professional decoupler
    GDP grew ~75% since 2003 (base year - consumption co2 data starts here)
    Until 2019: consumption CO2 ABOVE territorial - importing emissions
    After 2019: sharp synchronized drop in both curves
    *But we know it as a petrostate - not so easy to judge

Finding 6: The extreme cases - what the trade data exposes
   Malta (trade impact 708%):
    Tiny island economy, imports almost everything.
    Territorial CO2 means nothing here - 
    the country is almost entirely a consumption economy.
   
   Switzerland (trade impact 270%):
    Territorial CO2 dropped to 75 - looks impressively green.
    But consumption CO2 at 145 - nearly 3x territorial.
    The most dishonest 'green' reputation in Europe.
    Exports manufacturing, imports goods, hides emissions offshore.

Finding 7: Outliers - no or distorted decoupling
   Cyprus  - both CO2 curves grew WITH GDP (+52%).
             Tourism/services economy, high energy import dependency.
             No decoupling at all.
   
   Ireland - GDP index at 467 - but deeply misleading.
             Multinational HQs book profits in Ireland, 
             inflating GDP. Trade impact only 22% - 
             territorial decoupling is real, but GDP growth is not.
   
   Georgia - Soviet collapse masked as 'decoupling' in the 1990s.
             Territorial CO2 back near baseline, but consumption
             CO2 at 164 and rising fast (trade impact 29%).
             Classic recovering economy with growing 
             consumption and territorial CO2
""")

NOTEBOOK 04 - KEY FINDINGS

Finding 1: Territorial decoupling - widespread but uneven
   27/34 European countries show strong territorial decoupling.

   Most impressive performers (CO2 index, GDP index):
    Estonia:   CO2 (-68%), GDP (+48%)
    Lithuania: CO2 (-64%), GDP (+66%)
    Romania:   CO2 (-59%), GDP (+294%) - standout
    Slovakia:  CO2 (-49%), GDP (+131%)
    Sweden:    CO2 (-42%), GDP (+105%)

   Notable: Romania grew its economy nearly 4x while cutting
   emissions by 60% - the strongest combined performance in Europe.

Finding 2: Real vs Fake decoupling - the honest test
   Key metric: trade_co2_share - how much of consumption
   is actually imported emissions.

   Fake decouplers (consumption stayed high):
    Malta        - trade impact 709% - imports almost all consumption
    Switzerland  - trade impact 270% - most extreme large-country fake decoupler
    Belgium      - trade impact 132% - heavy outsourcing
    Belarus      - obvious faker, without doubt
    Denmark 

In [80]:
summary_data = {
    'Country': [
        # Genuine decouplers
        'Estonia', 'Lithuania', 'Romania', 'Slovakia',
        'Germany', 'Bulgaria', 'Sweden', 'Finland',
        # Poland - special case
        'Poland',
        # Fake decouplers
        'France', 'United Kingdom', 'Denmark',
        'Belgium', 'Switzerland', 'Malta', 'Belarus',
        # Late decoupler
        'Norway',
        # Outliers
        'Cyprus', 'Ireland', 'Georgia'
    ],
    'GDP Index': [
        148, 166, 394, 231,
        194, 174, 206, 169,
        396,
        163, 174, 198,
        179, 239, None, 150,
        175,
        323, 467, 107
    ],
    'Territorial CO2 Index': [
        32, 36, 41, 51,
        63, 61, 63, 64,
        84,
        75, 52, 54,
        74, 75, None, 53,
        95,
        152, 111, 81
    ],
    'Trade Impact %': [
        44, 64, 15, 14,
        29, 2, 68, 49,
        -3,
        51, 58, 72,
        132, 270, 709, 12,
        0.3,
        20, 22, 29
    ],
    'Verdict': [
        # Genuine
        '✅ Real decoupling', '✅ Real decoupling',
        '✅ Real decoupling', '✅ Real decoupling',
        '✅ Real decoupling', '✅ Real decoupling',
        '✅ Real decoupling', '✅ Real decoupling (from 2010)',
        # Poland
        '🟡 Stable emissions, net exporter',
        # Fake
        '⚠️ Partial fake (lagged 15y)', '⚠️ Partial fake (until 2010)',
        '⚠️ Fake decoupling',
        '⚠️ Fake decoupling', '⚠️ Most extreme fake', '⚠️ Island economy',
        '⚠️ Fake decoupling',
        # Late
        '✅ Late decoupler (from 2019)',
        # Outliers
        '❌ No decoupling', '⚠️ GDP inflated', '⚠️ Recovery + outsourcing'
    ]
}

summary_df = pd.DataFrame(summary_data)

# Sort by verdict category then territorial CO2
verdict_order = {
    '✅ Real decoupling': 0,
    '✅ Real decoupling (from 2010)': 1,
    '✅ Late decoupler (from 2019)': 2,
    '🟡 Stable emissions, net exporter': 3,
    '⚠️ Partial fake (lagged 15y)': 4,
    '⚠️ Partial fake (until 2010)': 5,
    '⚠️ Fake decoupling': 6,
    '⚠️ Most extreme fake': 7,
    '⚠️ Island economy': 8,
    '⚠️ GDP inflated': 9,
    '⚠️ Recovery + outsourcing': 10,
    '❌ No decoupling': 11,
}

summary_df['sort_key'] = summary_df['Verdict'].map(verdict_order)
summary_df = summary_df.sort_values(['sort_key', 'Territorial CO2 Index'])
summary_df = summary_df.drop('sort_key', axis=1)

print(summary_df.to_string(index=False))

       Country  GDP Index  Territorial CO2 Index  Trade Impact %                          Verdict
       Estonia      148.0                   32.0            44.0                ✅ Real decoupling
     Lithuania      166.0                   36.0            64.0                ✅ Real decoupling
       Romania      394.0                   41.0            15.0                ✅ Real decoupling
      Slovakia      231.0                   51.0            14.0                ✅ Real decoupling
      Bulgaria      174.0                   61.0             2.0                ✅ Real decoupling
       Germany      194.0                   63.0            29.0                ✅ Real decoupling
        Sweden      206.0                   63.0            68.0                ✅ Real decoupling
       Finland      169.0                   64.0            49.0    ✅ Real decoupling (from 2010)
        Norway      175.0                   95.0             0.3     ✅ Late decoupler (from 2019)
        Poland      